# 02 — Spike Detection

Reads pre-processed spike data from `data/processed/spikes.csv` (built by `process_data.py`) and explores spike distribution across companies and sectors.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 20)

## 1. Load data

In [ ]:
spike_df = pd.read_csv('../data/processed/spikes.csv', parse_dates=['date'])

n_spikes = spike_df['is_spike'].sum()
print(f"{n_spikes:,} spike days across {spike_df['article'].nunique()} articles")
print(f"Date range: {spike_df['date'].min().date()} to {spike_df['date'].max().date()}")
print(f"Spike rate: {n_spikes / len(spike_df):.1%}")
spike_df.head()

## 2. Spike rate by sector

In [ ]:
sector_stats = (
    spike_df.groupby('sector')
    .agg(total_days=('is_spike', 'count'), spike_days=('is_spike', 'sum'), n_companies=('ticker', 'nunique'))
    .assign(spike_rate=lambda d: d['spike_days'] / d['total_days'])
    .sort_values('spike_rate', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sector_stats.index, sector_stats['spike_rate'] * 100)
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_xlabel('Spike rate (%)')
ax.set_title('Wikipedia spike rate by sector (2-sigma threshold)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/spike_rate_by_sector.png', dpi=150)
plt.show()
sector_stats

## 3. Top 20 companies by spike count

In [ ]:
top_companies = (
    spike_df[spike_df['is_spike']]
    .groupby(['ticker', 'company', 'sector'])['is_spike']
    .sum().reset_index(name='spike_count')
    .sort_values('spike_count', ascending=False).head(20)
)

sectors = sorted(spike_df['sector'].dropna().unique())
sector_color = dict(zip(sectors, sns.color_palette('tab10', n_colors=len(sectors))))
bar_colors = [sector_color.get(s, 'gray') for s in top_companies['sector']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_companies['company'], top_companies['spike_count'], color=bar_colors)
ax.bar_label(bars, padding=3)
ax.set_xlabel('Number of spike days')
ax.set_title('Top 20 companies by Wikipedia spike count')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/top_companies_spike_count.png', dpi=150)
plt.show()
top_companies

## 4. Spike magnitude distribution

In [ ]:
spikes_only = spike_df[spike_df['is_spike']].copy()
spikes_only['spike_magnitude'] = (spikes_only['views'] - spikes_only['rolling_mean']) / spikes_only['rolling_std']

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(spikes_only['spike_magnitude'].clip(upper=20), bins=50, edgecolor='white')
ax.axvline(2, color='red', linestyle='--', label='2-sigma threshold')
ax.set_xlabel('Spike magnitude (sigma above rolling mean)')
ax.set_ylabel('Count')
ax.set_title('Distribution of spike magnitudes')
ax.legend()
plt.tight_layout()
plt.savefig('../results/spike_magnitude_distribution.png', dpi=150)
plt.show()
spikes_only['spike_magnitude'].describe()

## 5. Single company pageview timeline

Change `TICKER` to explore any company.

In [ ]:
TICKER = 'TSLA'

company_df = spike_df[spike_df['ticker'] == TICKER].copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(company_df['date'], company_df['views'], label='Daily views', lw=1)
ax.plot(company_df['date'], company_df['rolling_mean'], label='30d rolling mean', lw=1.5, linestyle='--')
ax.fill_between(
    company_df['date'],
    company_df['rolling_mean'] + 2 * company_df['rolling_std'],
    company_df['rolling_mean'].clip(lower=0),
    alpha=0.15, label='2-sigma band'
)
spike_rows = company_df[company_df['is_spike']]
ax.scatter(spike_rows['date'], spike_rows['views'], color='red', zorder=5, label='Spike', s=20)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.set_ylabel('Page views')
ax.set_title(f'Wikipedia pageviews - {TICKER}')
ax.legend()
plt.tight_layout()
plt.savefig(f'../results/timeline_{TICKER}.png', dpi=150)
plt.show()